In [ ]:
from pathlib import Path

# Encuentra la raíz del repositorio desde Jupyter, independientemente
# de si se inicia desde la raíz o desde analysis/notebooks/.
def repo_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "requirements.txt").is_file() and (folder / "analysis").is_dir():
            return folder
    raise FileNotFoundError("No se encontró Neutrino-Simulations. Inicia Jupyter dentro del repositorio.")

REPO = repo_root()
ROOT_DATA = REPO / "data" / "root"

def root_input(name):
    path = ROOT_DATA / name
    if not path.is_file():
        raise FileNotFoundError(f"Falta el archivo {path}. Súbelo a data/root/ o omite esta sección.")
    return path



In [ ]:
import re
import pandas as pd

filename = REPO / "data" / "honda" / "kam-nu-20-01-000.d"
if not filename.is_file():
    raise FileNotFoundError(f"Falta {filename}. Esta primera sección necesita el archivo Honda original.")

rows = []
current_cosz = None

with open(filename) as f:
    for line in f:

        line = line.strip()

        # detectar bloque cosZ
        m = re.search(r'cosZ\s*=\s*([-\d\.]+)\s*--\s*([-\d\.]+)', line)
        if m:
            z1 = float(m.group(1))
            z2 = float(m.group(2))
            current_cosz = 0.5*(z1+z2)
            continue

        # leer filas numéricas
        parts = line.split()

        if len(parts) == 5:
            try:
                E = float(parts[0])
                numu = float(parts[1])   # ajustar columna si necesario

                if current_cosz is not None and current_cosz < 0:
                    rows.append([E,current_cosz,numu])

            except:
                pass

df = pd.DataFrame(rows, columns=["Enu","cosz","numu"])

# sumar bins angulares
flux = df.groupby("Enu")["numu"].sum().reset_index()

# limitar a 1000 GeV
flux = flux[flux["Enu"] <= 1000]

# ordenar
flux = flux.sort_values("Enu")

flux.to_csv("up_numuflux_clean.csv", index=False)

print(flux.head())
print("Archivo creado: up_numuflux_clean.csv")

In [ ]:
flux

In [ ]:
import uproot

file = uproot.open(root_input("RHC_Flux_NOvA_ND_2017.root"))

# Listar objetos y sus clases ROOT
for name, classname in file.classnames().items():
    print(f"{name}: {classname}")


In [ ]:
# Acceder al objeto flux_numu (ciclo 1, como en el notebook original)
flux_numu = file["flux_numu;1"]
print("Clase:", flux_numu.classname)

# GetEntries() corresponde a un TTree; otros objetos (por ejemplo TH1)
# requieren métodos diferentes para contar sus entradas.
if flux_numu.classname.startswith("TTree") or "RNTuple" in flux_numu.classname:
    print("Entradas:", flux_numu.num_entries)
elif flux_numu.classname.startswith(("TH1", "TH2", "TH3")):
    print("Entradas del histograma:", flux_numu.member("fEntries"))
    print("Contenido de bins:", flux_numu.values())
else:
    print("Objeto disponible; inspecciona sus miembros con flux_numu.members")


In [ ]:
import uproot

f = uproot.open(root_input("gntp.0.ghep.1000_converted.root"))
print(f.keys())

In [ ]:
event = f["Event"]
particles = f["Particles"]

print(event.keys())
print(particles.keys())

In [ ]:
import awkward as ak

pp = particles.arrays(library="ak")
pp

In [ ]:
ev = event.arrays(library="pd")

In [ ]:
ev[["nuE","nuPx","nuPy","nuPz"]].head(10)

In [ ]:
ev["IsDIS"].mean()

In [ ]:
import matplotlib.pyplot as plt
plt.hist(ev["nuE"], bins=50, log=True)
plt.xlabel("Eν [GeV]")
plt.show()

In [ ]:
plt.hist(ev[ev["IsQE"]]["nuE"], bins=50, alpha=0.5, label="QE")
plt.hist(ev[ev["IsDIS"]]["nuE"], bins=50, alpha=0.5, label="DIS")
plt.xscale("log")
plt.legend()

In [ ]:
(ev["nuPx"] != 0).sum(), (ev["nuPy"] != 0).sum()

In [ ]:
import numpy as np 
pt = np.sqrt(ak.to_numpy(ak.flatten(pp["px"]**2 + pp["py"]**2)))

import matplotlib.pyplot as plt
plt.hist(pt, bins=50)
plt.xlabel("pT")
plt.ylabel("Entries")
plt.show()

In [ ]:
mask = (pp["status"] == 1)
px_final = pp["px"][mask]
py_final = pp["py"][mask]

In [ ]:
ak.sum(px_final != 0), ak.sum(py_final != 0)

In [ ]:
import awkward as ak

# todas las partículas finales
final_mask = (pp["status"] == 1)

# muones finales
mu_minus = final_mask & (pp["pdg"] == 13)
mu_plus  = final_mask & (pp["pdg"] == -13)

n_mu_minus_total = ak.sum(mu_minus)
n_mu_plus_total  = ak.sum(mu_plus)

print("mu- finales:", n_mu_minus_total)
print("mu+ finales:", n_mu_plus_total)

In [ ]:
events_with_mu_minus = ak.sum(ak.any(mu_minus, axis=1))
events_with_mu_plus  = ak.sum(ak.any(mu_plus, axis=1))

print("Eventos con al menos un mu-:", events_with_mu_minus)
print("Eventos con al menos un mu+:", events_with_mu_plus)

In [ ]:
pp = particles.arrays(library="ak")

In [ ]:
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

mu_mask = (pp["status"] == 1) & (pp["pdg"] == 13) # status ==1 particuals finales

In [ ]:
muE = ak.to_numpy(ak.flatten(pp["energy"][mu_mask]))

plt.figure(figsize=(8,5))
plt.hist(muE, bins=50, histtype="step", linewidth=2)
plt.xlabel(r"$E_{\mu^-}$ [GeV]")
plt.ylabel("Counts")
plt.title("Muon final-state energy")
plt.yscale("log")
plt.show()

In [ ]:
px = pp["px"][mu_mask]
py = pp["py"][mu_mask]

pt = ak.to_numpy(ak.flatten(np.sqrt(px**2 + py**2)))

plt.figure(figsize=(8,5))
plt.hist(pt, bins=50, histtype="step", linewidth=2)
plt.xlabel(r"$p_T^\mu$ [GeV]")
plt.ylabel("Counts")
plt.title("Muon transverse momentum")
plt.yscale("log")
plt.show()

In [ ]:
pz = pp["pz"][mu_mask]
p = np.sqrt(px**2 + py**2 + pz**2)

costh = ak.to_numpy(ak.flatten(pz/p))

plt.figure(figsize=(8,5))
plt.hist(costh, bins=50, histtype="step", linewidth=2)
plt.xlabel(r"$\cos\theta_\mu$")
plt.ylabel("Counts")
plt.title("Muon angular distribution")
plt.show()

In [ ]:
mu_mask = (pp["status"] == 1) & (pp["pdg"] == 13)
muE = ak.to_numpy(ak.flatten(pp["energy"][mu_mask]))

plt.hist(muE, bins=50)
plt.xlabel("Muon Energy [GeV]")
plt.show()

In [ ]:
dis = ev[ev['IsDIS'] == True]
dis

In [ ]:
import numpy as np
import pandas as pd

bins = np.logspace(np.log10(dis['nuE'].min()), np.log10(dis['nuE'].max()), 20)

dis2 = dis.copy()
dis2["bin"] = pd.cut(dis2["nuE"], bins)

grp = dis2.groupby("bin").agg({
    "nuE":"mean",
    "xsection":"mean"
}).dropna()

plt.figure(figsize=(8,5))
plt.plot(grp["nuE"], grp["xsection"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$E_\nu$ [GeV]")
plt.ylabel(r"$\langle \sigma \rangle$")
plt.title("Average DIS cross section")
plt.show()

In [ ]:
dis_numu_cc = ev[
    (ev["nupdg"] == 14) &
    (ev["IsDIS"] == True) &
    (ev["IsCC"] == True)
].copy()

In [ ]:
len(dis_numu_cc)


In [ ]:
dis_numu_cc[["nupdg", "nuE", "IsDIS", "IsCC", "IsNC"]].head()

In [ ]:
print(ev.columns)
print(pp.fields)

In [ ]:
dis_numu_cc

In [ ]:
bins = np.logspace(np.log10(dis_numu_cc['nuE'].min()), np.log10(dis_numu_cc['nuE'].max()), 20)

dis2 = dis_numu_cc.copy()
dis2["bin"] = pd.cut(dis2["nuE"], bins)

grp = dis2.groupby("bin").agg({
    "nuE":"mean",
    "xsection":"mean"
}).dropna()

plt.figure(figsize=(8,5))
plt.plot(grp["nuE"], grp["xsection"], marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$E_\nu$ [GeV]")
plt.ylabel(r"$\langle \sigma \rangle$")
plt.title("Average DIS cross section")
plt.show()

In [ ]:
ev["nupdg"].value_counts()

In [ ]:
print(ev.columns)

### A continuación abriré el output de GENIE para visualizar la información de los muones que salieron

In [ ]:
pip install uproot awkward pandas numpy

In [ ]:
import uproot

file = uproot.open(root_input("gntp.0.ghep.1000_converted.root"))

print(file.keys())

In [ ]:
particles = file['Particles']
event = file['Event']


In [ ]:
print(particles.keys())

In [ ]:
print(event.keys())

In [ ]:
import awkward as ak

arr = particles.arrays(library="ak")

pdg = arr["pdg"]

mu_mask = (pdg == 13) | (pdg == -13)

muons = arr[mu_mask]


In [ ]:
muons

In [ ]:

import awkward as ak

E_mu = ak.flatten(muons.energy)

print(E_mu)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(E_mu, bins=50)
plt.xlabel("Muon Energy [GeV]")
plt.ylabel("Counts")
plt.show()

In [ ]:
has_muon = ak.num(muons.energy) > 0

events_with_muon = arr[has_muon]

In [ ]:
events_with_muon

In [ ]:
import awkward as ak
import pandas as pd
import numpy as np

# muons ya viene de:
# muons = arr[mu_mask]

# -----------------------------------
# Flatten de todas las ramas
# -----------------------------------
pdg    = ak.to_numpy(ak.flatten(muons.pdg))
status = ak.to_numpy(ak.flatten(muons.status))
E      = ak.to_numpy(ak.flatten(muons.energy))
px     = ak.to_numpy(ak.flatten(muons.px))
py     = ak.to_numpy(ak.flatten(muons.py))
pz     = ak.to_numpy(ak.flatten(muons.pz))

# -----------------------------------
# Momento total
# -----------------------------------
p = np.sqrt(px**2 + py**2 + pz**2)

# cos(theta)
costheta = pz / p

# theta en grados
theta = np.degrees(np.arccos(costheta))

# -----------------------------------
# Crear DataFrame
# -----------------------------------
df_muons = pd.DataFrame({
    "pdg": pdg,
    "status": status,
    "energy_GeV": E,
    "px_GeV": px,
    "py_GeV": py,
    "pz_GeV": pz,
    "p_GeV": p,
    "costheta": costheta,
    "theta_deg": theta
})

print(df_muons.head(20))


# -----------------------------------
# Guardar dataset en CSV
# -----------------------------------
output_file = "muons_dataset.csv"

df_muons.to_csv(output_file, index=False)

print(f"Dataset guardado en: {output_file}")
print(f"Número de muones guardados: {len(df_muons)}")

In [ ]:
df_muons

#### Hasta ahora tengo el dataset de los muones, sin embargo necesitamos hacer un merge con el árbol 'events' para agregarle la información de isDIS y isCC.

In [ ]:
print(particles.keys())

In [ ]:
print(event.keys())

In [ ]:
evt = event.arrays(library="ak")

# muones flattened
df_mu = pd.DataFrame({
    "event": np.repeat(np.arange(len(muons)), ak.num(muons.pdg)),
    "pdg": ak.to_numpy(ak.flatten(muons.pdg)),
    "E": ak.to_numpy(ak.flatten(muons.energy))
})

# tabla de eventos
df_evt = pd.DataFrame({
    "event": np.arange(len(evt)),
    "isCC": ak.to_numpy(evt.IsCC),
    "isNC": ak.to_numpy(evt.IsNC),
    "isQE": ak.to_numpy(evt.IsQE),
    "isDIS": ak.to_numpy(evt.IsDIS),
    "isRES": ak.to_numpy(evt.IsRES)
})

# unir
df = df_mu.merge(df_evt, on="event")

In [ ]:
df

In [ ]:
import awkward as ak
import pandas as pd
import numpy as np

# =====================================================
# 1. Leer árbol Event
# =====================================================
evt = event.arrays(library="ak")

# =====================================================
# 2. Número de muones por evento
# =====================================================
nmu = ak.num(muons.pdg)

# =====================================================
# 3. Variables de muones (flatten)
# =====================================================
pdg    = ak.to_numpy(ak.flatten(muons.pdg))
status = ak.to_numpy(ak.flatten(muons.status))
E      = ak.to_numpy(ak.flatten(muons.energy))
px     = ak.to_numpy(ak.flatten(muons.px))
py     = ak.to_numpy(ak.flatten(muons.py))
pz     = ak.to_numpy(ak.flatten(muons.pz))

# =====================================================
# 4. Índice de evento repetido para cada muón
# =====================================================
event_id = np.repeat(np.arange(len(muons)), ak.to_numpy(nmu))

# =====================================================
# 5. Cinemática extra
# =====================================================
p = np.sqrt(px**2 + py**2 + pz**2) # Momentum total
costheta = pz / p
theta = np.degrees(np.arccos(costheta)) # angulo zenital?

# =====================================================
# 6. Crear tabla base de muones
# =====================================================
df_muons = pd.DataFrame({
    "event": event_id,
    "pdg": pdg,
    "status": status,
    "energy_GeV": E,
    "px_GeV": px,
    "py_GeV": py,
    "pz_GeV": pz,
    "p_GeV": p,
    "costheta": costheta,
    "theta_deg": theta
})

# =====================================================
# 7. Agregar flags del evento
# =====================================================
df_muons["IsCC"]  = ak.to_numpy(evt["IsCC"])[event_id]
df_muons["IsNC"]  = ak.to_numpy(evt["IsNC"])[event_id]
df_muons["IsQE"]  = ak.to_numpy(evt["IsQE"])[event_id]
df_muons["IsDIS"] = ak.to_numpy(evt["IsDIS"])[event_id]
df_muons["IsRES"] = ak.to_numpy(evt["IsRES"])[event_id]
df_muons["IsMEC"] = ak.to_numpy(evt["IsMEC"])[event_id]

# =====================================================
# 8. Mostrar resultado
# =====================================================
print(df_muons.head(20))

In [ ]:
df_muons_dis = df_muons[
    (df_muons['IsDIS'] == True) &
    (df_muons['IsCC']  == True)
].copy()

In [ ]:
df_muons_dis

#### Archivo con n = 10.000

In [ ]:
import uproot
import awkward as ak


file = uproot.open(root_input("gntp.fluxnumu_O16-104_converted.root"))

print(file.keys())

In [ ]:
particles = file['Particles']
event = file['Event']

print(particles.keys(),event.keys())

In [ ]:
arr = particles.arrays(library="ak")

pdg = arr["pdg"]

mu_mask = (pdg == 13) | (pdg == -13)

muons = arr[mu_mask]


In [ ]:
muons

In [ ]:
import awkward as ak
import pandas as pd
import numpy as np

# muons ya viene de:
# muons = arr[mu_mask]

# -----------------------------------
# Flatten de todas las ramas
# -----------------------------------
pdg    = ak.to_numpy(ak.flatten(muons.pdg))
status = ak.to_numpy(ak.flatten(muons.status))
E      = ak.to_numpy(ak.flatten(muons.energy))
px     = ak.to_numpy(ak.flatten(muons.px))
py     = ak.to_numpy(ak.flatten(muons.py))
pz     = ak.to_numpy(ak.flatten(muons.pz))

# -----------------------------------
# Momento total
# -----------------------------------
p = np.sqrt(px**2 + py**2 + pz**2)

# cos(theta)
costheta = pz / p

# theta en grados
theta = np.degrees(np.arccos(costheta))

# -----------------------------------
# Crear DataFrame
# -----------------------------------
df_muons = pd.DataFrame({
    "pdg": pdg,
    "status": status,
    "energy_GeV": E,
    "px_GeV": px,
    "py_GeV": py,
    "pz_GeV": pz,
    "p_GeV": p,
    "costheta": costheta,
    "theta_deg": theta
})

print(df_muons.head(20))

In [ ]:
import awkward as ak
import pandas as pd
import numpy as np

# =====================================================
# 1. Leer árbol Event
# =====================================================
evt = event.arrays(library="ak")

# =====================================================
# 2. Número de muones por evento
# =====================================================
nmu = ak.num(muons.pdg)

# =====================================================
# 3. Variables de muones (flatten)
# =====================================================
pdg    = ak.to_numpy(ak.flatten(muons.pdg))
status = ak.to_numpy(ak.flatten(muons.status))
E      = ak.to_numpy(ak.flatten(muons.energy))
px     = ak.to_numpy(ak.flatten(muons.px))
py     = ak.to_numpy(ak.flatten(muons.py))
pz     = ak.to_numpy(ak.flatten(muons.pz))

# =====================================================
# 4. Índice de evento repetido para cada muón
# =====================================================
event_id = np.repeat(np.arange(len(muons)), ak.to_numpy(nmu))

# =====================================================
# 5. Cinemática extra
# =====================================================
p = np.sqrt(px**2 + py**2 + pz**2) # Momentum total
costheta = pz / p
theta = np.degrees(np.arccos(costheta)) # angulo zenital?

# =====================================================
# 6. Crear tabla base de muones
# =====================================================
df_muons = pd.DataFrame({
    "event": event_id,
    "pdg": pdg,
    "status": status,
    "energy_GeV": E,
    "px_GeV": px,
    "py_GeV": py,
    "pz_GeV": pz,
    "p_GeV": p,
    "costheta": costheta,
    "theta_deg": theta
})

# =====================================================
# 7. Agregar flags del evento
# =====================================================
df_muons["IsCC"]  = ak.to_numpy(evt["IsCC"])[event_id]
df_muons["IsNC"]  = ak.to_numpy(evt["IsNC"])[event_id]
df_muons["IsQE"]  = ak.to_numpy(evt["IsQE"])[event_id]
df_muons["IsDIS"] = ak.to_numpy(evt["IsDIS"])[event_id]
df_muons["IsRES"] = ak.to_numpy(evt["IsRES"])[event_id]
df_muons["IsMEC"] = ak.to_numpy(evt["IsMEC"])[event_id]

# =====================================================
# 8. Mostrar resultado
# =====================================================
print(df_muons.head(20))

In [ ]:
df_muons

In [ ]:
df_muons_dis = df_muons[
    (df_muons['IsDIS'] == True) &
    (df_muons['IsCC']  == True)
].copy()

In [ ]:
df_muons_dis

In [ ]:

# Guardar CSV
df_muons_dis.to_csv("muon_data.csv", index=False)

print(df_muons_dis.head())
print(f"\nArchivo guardado: puntos_volcan_lascar.csv")
print(f"Total puntos: {len(df)}")

In [ ]:
df_muons_dis

In [ ]:
import matplotlib.pyplot as plt 

plt.hist(df_muons_dis['energy_GeV'])
plt.xlabel("Muon energy [GeV]")
plt.show()

## Muon angular distribution

In [ ]:
plt.hist(df_muons_dis["theta_deg"], bins=50)
plt.show()

In [ ]:
plt.hist2d(df_muons_dis["theta_deg"], df_muons_dis["energy_GeV"], bins=50)
plt.yscale("log")
plt.ylabel('Energy [GeV]')
plt.xlabel(r'$\theta$ [degrees]')
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# GENERADOR DE PUNTOS EN UN CONO TRUNCADO (Volcán Lascar aprox.)
# ============================================================
# Sistema de coordenadas:
# z = altura vertical
# eje del cono en z
# centro en (0,0)
#
# x,y en metros
# z en metros
#
# ------------------------------------------------------------
# PARÁMETROS GEOMÉTRICOS (AJUSTABLES)
# Valores aproximados para Lascar
# ------------------------------------------------------------

H = 1800          # altura del cono truncado [m]
R_base = 3000     # radio en la base [m]
R_top = 400       # radio en la cima truncada [m]

# Cantidad total de puntos deseados
N = 50000

# ------------------------------------------------------------
# CONTROL DE DENSIDAD VERTICAL
# alpha = 1   -> uniforme en volumen
# alpha > 1   -> más puntos abajo
# alpha < 1   -> más puntos arriba
# Recomiendo alpha=2 o 3
# ------------------------------------------------------------

alpha = 2.5

# ============================================================
# FUNCIÓN RADIO(z)
# ============================================================

def radius(z):
    """
    Radio del cono truncado a altura z
    z=0 base
    z=H cima
    """
    return R_base + (R_top - R_base) * (z / H)

# ============================================================
# MUESTREO EN z
# ============================================================
# Variable u uniforme [0,1]
# Transformación para concentrar puntos abajo:
# z = H * u^(alpha)

u = np.random.rand(N)
z = H * (u ** alpha)

# ============================================================
# MUESTREO UNIFORME EN CADA DISCO TRANSVERSAL
# ============================================================

rmax = radius(z)

theta = 2 * np.pi * np.random.rand(N)

# sqrt para distribución uniforme en área
r = rmax * np.sqrt(np.random.rand(N))

x = r * np.cos(theta)
y = r * np.sin(theta)

# ============================================================
# TABLA FINAL
# ============================================================

df = pd.DataFrame({
    "x_m": x,
    "y_m": y,
    "z_m": z
})

# Guardar CSV
df.to_csv("puntos_volcan_lascar.csv", index=False)

print(df.head())
print(f"\nArchivo guardado: puntos_volcan_lascar.csv")
print(f"Total puntos: {len(df)}")

In [ ]:
import numpy as np
import pandas as pd

# ==========================================================
# LEER TUS MUONES
# ==========================================================
df = pd.read_csv("muon_data.csv")

# ==========================================================
# GEOMETRÍA VOLCÁN LÁSCAR (ajustable)
# ==========================================================
H = 1800.0       # altura [m]
R_base = 3000.0  # radio base [m]
R_top = 400.0    # radio cima [m]

# ==========================================================
# FUNCIÓN RADIO(z)
# ==========================================================
def radius(z):
    return R_base + (R_top - R_base) * (z / H)

# ==========================================================
# VERIFICAR SI UN PUNTO ESTÁ DENTRO DEL CONO
# ==========================================================
def inside_volcano(x, y, z):

    if z < 0 or z > H:
        return False

    r = np.sqrt(x**2 + y**2)
    return r <= radius(z)

# ==========================================================
# NORMALIZAR DIRECCIONES DE MUONES
# ==========================================================
p = np.sqrt(df["px_GeV"]**2 + df["py_GeV"]**2 + df["pz_GeV"]**2)

df["ux"] = df["px_GeV"] / p
df["uy"] = df["py_GeV"] / p
df["uz"] = df["pz_GeV"] / p

# ==========================================================
# GENERAR PUNTOS DENTRO DEL VOLCÁN USANDO TRAYECTORIAS
# ==========================================================
points = []

N_steps = 200        # pasos por muón
step_size = 20.0     # metros

for _, row in df.iterrows():

    ux, uy, uz = row["ux"], row["uy"], row["uz"]

    # Solo muones descendentes
    if uz >= 0:
        continue

    # Punto inicial aleatorio en tapa superior
    theta = np.random.uniform(0, 2*np.pi)
    r = R_top * np.sqrt(np.random.rand())

    x0 = r*np.cos(theta)
    y0 = r*np.sin(theta)
    z0 = H

    for i in range(N_steps):

        s = i * step_size

        x = x0 + ux*s
        y = y0 + uy*s
        z = z0 + uz*s

        if inside_volcano(x, y, z):
            points.append([x, y, z, row["energy_GeV"]])
        else:
            break

# ==========================================================
# GUARDAR RESULTADO
# ==========================================================
out = pd.DataFrame(points, columns=["x_m", "y_m", "z_m", "energy_GeV"])

out.to_csv("puntos_muones_lascar.csv", index=False)

print(out.head())
print("Total puntos:", len(out))

In [ ]:
import numpy as np
import pandas as pd

# ==========================================================
# LEER DATASET ORIGINAL DE MUONES
# ==========================================================
df = pd.read_csv("muon_data.csv")

# ==========================================================
# GEOMETRÍA VOLCÁN LÁSCAR (ajustable)
# ==========================================================
H = 1800.0
R_base = 3000.0
R_top = 400.0

# ==========================================================
# FUNCIÓN RADIO(z)
# ==========================================================
def radius(z):
    return R_base + (R_top - R_base) * (z / H)

# ==========================================================
# VERIFICAR SI UN PUNTO ESTÁ DENTRO DEL VOLCÁN
# ==========================================================
def inside_volcano(x, y, z):
    if z < 0 or z > H:
        return False

    r = np.sqrt(x**2 + y**2)
    return r <= radius(z)

# ==========================================================
# NORMALIZAR MOMENTO -> VECTOR DIRECCIÓN
# ==========================================================
p = np.sqrt(df["px_GeV"]**2 + df["py_GeV"]**2 + df["pz_GeV"]**2)

df["ux"] = df["px_GeV"] / p
df["uy"] = df["py_GeV"] / p
df["uz"] = df["pz_GeV"] / p

# ==========================================================
# GENERAR NUEVO DATASET CONSERVANDO TODA LA INFO ORIGINAL
# ==========================================================
rows = []

N_steps = 200
step_size = 20.0

for _, row in df.iterrows():

    ux, uy, uz = row["ux"], row["uy"], row["uz"]

    # solo muones descendentes
    if uz >= 0:
        continue

    # entrada aleatoria por tapa superior
    theta = np.random.uniform(0, 2*np.pi)
    r = R_top * np.sqrt(np.random.rand())

    x0 = r*np.cos(theta)
    y0 = r*np.sin(theta)
    z0 = H

    for i in range(N_steps):

        s = i * step_size

        x = x0 + ux*s
        y = y0 + uy*s
        z = z0 + uz*s

        if inside_volcano(x, y, z):

            # copiar toda la fila original
            new_row = row.to_dict()

            # agregar posiciones
            new_row["x_m"] = x
            new_row["y_m"] = y
            new_row["z_m"] = z
            new_row["step_id"] = i

            rows.append(new_row)

        else:
            break

# ==========================================================
# DATAFRAME FINAL
# ==========================================================
out = pd.DataFrame(rows)

# quitar columnas auxiliares si no las quieres
out = out.drop(columns=["ux", "uy", "uz"])

# guardar
out.to_csv("puntos_muones_lascar_full.csv", index=False)

print(out.head())
print("Filas finales:", len(out))
print("Columnas:", out.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd

# ======================================================
# LEER MUONES
# ======================================================
df = pd.read_csv("muon_data.csv")

# ======================================================
# VOLCÁN LÁSCAR
# ======================================================
H = 1800.0
R_base = 3000.0
R_top = 400.0

# ======================================================
def radius(z):
    return R_base + (R_top - R_base)*(z/H)

def inside_volcano(x,y,z):

    if z < 0 or z > H:
        return False

    r = np.sqrt(x**2 + y**2)
    return r <= radius(z)

# ======================================================
# NORMALIZAR DIRECCIONES
# ======================================================
p = np.sqrt(df["px_GeV"]**2 + df["py_GeV"]**2 + df["pz_GeV"]**2)

df["ux"] = df["px_GeV"]/p
df["uy"] = df["py_GeV"]/p
df["uz"] = df["pz_GeV"]/p

# ======================================================
# GENERAR VÉRTICES + TRACKS
# ======================================================
rows = []

N_steps = 200
step_size = 15.0

for _, row in df.iterrows():

    ux, uy, uz = row["ux"], row["uy"], row["uz"]

    # ----------------------------------------------
    # VÉRTICE INTERNO uniforme en volumen
    # ----------------------------------------------
    z0 = H*np.random.rand()

    rmax = radius(z0)

    rr = rmax*np.sqrt(np.random.rand())
    phi = 2*np.pi*np.random.rand()

    x0 = rr*np.cos(phi)
    y0 = rr*np.sin(phi)

    # ----------------------------------------------
    # PROPAGACIÓN DEL MUÓN
    # ----------------------------------------------
    for i in range(N_steps):

        s = i*step_size

        x = x0 + ux*s
        y = y0 + uy*s
        z = z0 + uz*s

        if inside_volcano(x,y,z):

            new_row = row.to_dict()

            # vértice interacción
            new_row["x_vertex"] = x0
            new_row["y_vertex"] = y0
            new_row["z_vertex"] = z0

            # track muón
            new_row["x_m"] = x
            new_row["y_m"] = y
            new_row["z_m"] = z

            new_row["step_id"] = i

            rows.append(new_row)

        else:
            break

# ======================================================
# OUTPUT
# ======================================================
out = pd.DataFrame(rows)

out.drop(columns=["ux","uy","uz"], inplace=True)

out.to_csv("muon_data_with_positions.csv", index=False)

print(out.head())
print("Total filas:", len(out))

In [ ]:
out

In [ ]:
import pandas as pd

# -------------------------------------------------
# TABLA 1: SOLO MUONES ÚNICOS
# -------------------------------------------------
muons = out.drop_duplicates(subset=["event"]).copy()

muons.to_csv("muons_clean.csv", index=False)

# -------------------------------------------------
# TABLA 2: SOLO POSICIONES
# -------------------------------------------------
tracks = out[["event","step_id","x_m","y_m","z_m"]].copy()

tracks.to_csv("tracks_clean.csv", index=False)

print("Muones únicos:", len(muons))
print("Puntos trayectoria:", len(tracks))

In [ ]:
muons

In [ ]:
tracks

In [ ]:
import numpy as np
import pandas as pd

# ======================================
# Leer datos
# ======================================
df = pd.read_csv("muon_data.csv")

# ======================================
# Geometría Lascar
# ======================================
H = 1800.0
R_base = 3000.0
R_top  = 400.0

def radius(z):
    return R_base + (R_top - R_base)*(z/H)

def inside_volcano(x,y,z):
    if z < 0 or z > H:
        return False
    return np.sqrt(x*x+y*y) <= radius(z)

# ======================================
# Dirección muón
# ======================================
p = np.sqrt(df["px_GeV"]**2 + df["py_GeV"]**2 + df["pz_GeV"]**2)

df["ux"] = df["px_GeV"]/p
df["uy"] = df["py_GeV"]/p
df["uz"] = df["pz_GeV"]/p

# ======================================
# Buscar vértices consistentes
# ======================================
xv, yv, zv = [], [], []

for _, row in df.iterrows():

    ux, uy, uz = row["ux"], row["uy"], row["uz"]

    # punto de salida asumido en cima (aprox)
    phi = np.random.uniform(0,2*np.pi)
    rr  = R_top*np.sqrt(np.random.rand())

    x_exit = rr*np.cos(phi)
    y_exit = rr*np.sin(phi)
    z_exit = H

    # distancia retrocedida dentro del volcán
    L = np.random.uniform(50,800)  # metros

    x0 = x_exit - L*ux
    y0 = y_exit - L*uy
    z0 = z_exit - L*uz

    # si cae fuera, reubicar simple
    if not inside_volcano(x0,y0,z0):
        x0, y0, z0 = np.nan, np.nan, np.nan

    xv.append(x0)
    yv.append(y0)
    zv.append(z0)

# ======================================
# Guardar
# ======================================
df["x_vertex_m"] = xv
df["y_vertex_m"] = yv
df["z_vertex_m"] = zv

df.drop(columns=["ux","uy","uz"], inplace=True)

df.to_csv("muon_data_vertices_momentum.csv", index=False)

print(df.head())

In [ ]:
df

In [ ]:
df.info()